# Week 1 — Single Replay Exploration (mgz-fast)

**Goal:** Parse one personal replay with mgz-fast, print every extractable field, and document what each feature means in AOE2 terms.

**Why mgz-fast instead of AgeAlyser_2:**
AgeAlyser_2's `advanced_parser()` did not expose `game_result` (win/loss) as a direct pandas field — it was buried in `match_json['players'][n]['winner']` which was unreliable. AgeAlyser_2 also threw `ERROR: Technology could not be found in Enums` for many newer DLC techs (Chieftains, Savar, Kamandaran, etc.), inflating the failure rate.

mgz-fast (`pip install mgz-fast`, installs as the `mgz` package v1.8.51) provides:
- `de['rated']` — direct boolean for ranked game filter
- `de['rms_map_id']` — numeric map ID (Arabia = 9)
- RESIGN action → loser player_id (winner = the other player)
- RESEARCH actions with tech IDs 101/102/103 → feudal/castle/imperial age-up times
- POSTGAME block → leaderboard ratings (Elo) per player
- SYNC increments → game duration in milliseconds

**Import:** `from mgz.fast import header as fast_header, operation, Operation, Action, meta`

**Body parsing note:** After `fast_header.parse(f)`, always call `meta(f)` before scanning operations.
The body starts with a metadata preamble; skipping it correctly is required or the first `operation()` call fails.

**Tasks covered here:**
- Task 1: Parse 1 replay, extract all fields
- Task 2: Document every field — name, type, AOE2 meaning, expected range
- Task 3: Confirm mgz-fast output is sufficient for all required features

## Setup

In [38]:
import struct
import pandas as pd
import numpy as np
from pathlib import Path
from mgz.fast import header as fast_header, operation, Operation, Action, meta

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)

# AOE2 DE constants (verified from survey of personal replays)
ARABIA_MAP_ID = 9
AGE_UP_TECH_IDS = {101: 'feudal', 102: 'castle', 103: 'imperial'}
RM_1V1_LEADERBOARD_ID = 3  # Random Map 1v1 leaderboard

print('Imports OK')

Imports OK


## Section 1 — Find Replay Files

Scan the personal replay directory to confirm files exist and pick one for exploration.

In [39]:
REPLAY_DIR = Path('C:/Users/liher/Games/Age of Empires 2 DE/76561198151543542/savegame')

replay_files = sorted(REPLAY_DIR.glob('*.aoe2record'))
print(f'Found {len(replay_files)} replay files')
print()
print('Most recent 10 files:')
for f in reversed(replay_files[-10:]):
    print(f'  {f.name}')

Found 3627 replay files

Most recent 10 files:
  rec.aoe2record
  MP Replay v101.103.44206.0 @2026.05.09 210443 (2).aoe2record
  MP Replay v101.103.44206.0 @2026.05.06 010743 (8).aoe2record
  MP Replay v101.103.44206.0 @2026.05.04 000710 (8).aoe2record
  MP Replay v101.103.44206.0 @2026.05.03 231927 (4).aoe2record
  MP Replay v101.103.44206.0 @2026.05.03 221022 (8).aoe2record
  MP Replay v101.103.44206.0 @2026.05.03 212357 (8).aoe2record
  MP Replay v101.103.44206.0 @2026.05.03 204310 (7).aoe2record
  MP Replay v101.103.44206.0 @2026.05.03 191932 (4).aoe2record
  MP Replay v101.103.39862.0 @2026.03.28 203350 (5).aoe2record


## Section 2 — Parse a Single Replay

Finds the most recent rated 1v1 Arabia game in the personal replay directory.

**Parsing flow:**
1. `fast_header.parse(f)` — reads and decompresses the replay header; returns a dict with player info, map, DE settings
2. `meta(f)` — skips the body preamble (required before scanning operations)
3. Loop `operation(f)` — scans body operations for SYNC (timing), RESEARCH (age-ups), RESIGN (winner), POSTGAME (ratings)

In [49]:
# This is what the actual parsed header looks like. Note that the "de" field is a dict of all the DE-specific fields, 
# which are not standardized and can vary between replays.
import pprint
with open(replay_files[-1], 'rb') as f:
    h = fast_header.parse(f)
pprint.pprint(h)

'''with open(replay_files[-1], 'rb') as f:
    fast_header.parse(f)
    meta(f)
    while True:
        op_type, payload = operation(f)
        if op_type == Operation.ACTION:
            pprint.pprint(payload)
            break'''
            


(<Action.GAME: 103>, {'command_id': 16, 'player_id': 1, 'sequence': 416})


In [ ]:
def parse_replay(filepath):
    """Parse one .aoe2record file. Returns a dict of all extracted fields, or raises on failure."""
    with open(filepath, 'rb') as f:
        # --- Header ---
        h = fast_header.parse(f)
        de = h.get('de') or {}
        
        # Players (skip Gaia = player number 0)
        header_players = {p['number']: p for p in (h.get('players') or []) if p and p['type'] == 1}
        de_players = {p['number']: p for p in (de.get('players') or []) if p.get('number', -1) >= 1}
        
        # Map + game settings from DE section
        map_id = de.get('rms_map_id')
        rated = de.get('rated', False)
        
        # --- Body scan ---
        meta(f)  # skip body preamble — required before operation()
        
        game_time_ms = 0
        age_up_times = {}    # player_num -> {'feudal': min, 'castle': min, 'imperial': min}
        resign_player = None # player_num of the player who resigned (= loser)
        resign_time_min = None
        leaderboard_ratings = {}  # player_num -> rating
        world_time_ms = None
        
        try:
            while True:
                op_type, payload = operation(f)
                
                if op_type == Operation.SYNC:
                    increment, _, _ = payload
                    game_time_ms += increment
                
                elif op_type == Operation.ACTION:
                    action_type, ap = payload
                    t_min = game_time_ms / 60000
                    
                    if action_type == Action.RESIGN:
                        pid = ap.get('player_id')
                        if resign_player is None:  # first resign = the one that ends the game
                            resign_player = pid
                            resign_time_min = round(t_min, 2)
                    
                    elif action_type == Action.RESEARCH:
                        tech = ap.get('technology_id')
                        pid = ap.get('player_id')
                        if tech in AGE_UP_TECH_IDS:
                            age = AGE_UP_TECH_IDS[tech]
                            age_up_times.setdefault(pid, {})
                            if age not in age_up_times[pid]:
                                age_up_times[pid][age] = round(t_min, 2)
                
                elif op_type == Operation.POSTGAME:
                    world_time_ms = payload.get('world_time')
                    for lb in (payload.get('leaderboards') or []):
                        if lb.get('id') == RM_1V1_LEADERBOARD_ID:
                            for entry in lb.get('players') or []:
                                leaderboard_ratings[entry['number']] = entry['rating']
                    break
        except (EOFError, struct.error):
            pass
        
        # Duration: prefer authoritative world_time from POSTGAME; fall back to SYNC sum
        duration_min = round((world_time_ms or game_time_ms) / 60000, 2)
        
        # Diplomacy type: 1v1 = exactly 2 real players with opposing teams
        num_real_players = len(header_players)
        
        return {
            'filepath': str(filepath),
            'map_id': map_id,
            'rated': rated,
            'num_players': num_real_players,
            'duration_min': duration_min,
            'resign_player': resign_player,
            'resign_time_min': resign_time_min,
            'age_up_times': age_up_times,          # {player_num: {'feudal': m, 'castle': m, 'imperial': m}}
            'leaderboard_ratings': leaderboard_ratings,  # {player_num: elo}
            'header_players': header_players,      # {player_num: {...}}
            'de_players': de_players,              # {player_num: {...}}
        }


# Find most recent rated 1v1 Arabia game > 8 minutes long, and parse it. Print any parse failures at the end.
result = None
failed = []
REPLAY_PATH = None

for filepath in reversed(replay_files):
    try:
        r = parse_replay(filepath)
        if r['map_id'] == ARABIA_MAP_ID and r['rated'] and r['num_players'] == 2 and r['duration_min'] > 8:
            result = r
            REPLAY_PATH = filepath
            print(f'SUCCESS: {filepath.name}')
            break
    except Exception as e:
        failed.append((filepath.name, f'{type(e).__name__}: {e}'))

if result is None:
    print('No rated 1v1 Arabia game found in most recent replays.')
    if failed:
        print(f'Parse failures: {len(failed)}')
        for name, reason in failed[:5]:
            print(f'  FAIL: {name} — {reason}')

SUCCESS: MP Replay v101.103.31214.0 @2026.02.17 221325 (1).aoe2record


## Section 3 — Print Full Extracted Output

Print every field returned by `parse_replay()`. This is the canonical field reference for all modeling decisions in Weeks 2–3.

In [4]:
print('=== Game-level fields ===')
for k in ['filepath', 'map_id', 'rated', 'num_players', 'duration_min', 'resign_player', 'resign_time_min']:
    print(f'  {k:25s}: {result[k]}')

print()
print('=== Age-up times (player_num -> age -> minutes) ===')
for pid, times in result['age_up_times'].items():
    print(f'  Player {pid}: {times}')

print()
print('=== Leaderboard ratings (player_num -> Elo) ===')
for pid, rating in result['leaderboard_ratings'].items():
    print(f'  player_num={pid}: {rating}')

print()
print('=== Header players ===')
for pnum, p in result['header_players'].items():
    safe = {k: v for k, v in p.items() if k not in ('objects', 'sleeping', 'doppel')}
    print(f'  Player {pnum}: {safe}')

print()
print('=== DE players ===')
for pnum, p in result['de_players'].items():
    print(f'  Player {pnum}: {p}')

=== Game-level fields ===
  filepath                 : C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @2026.02.17 221325 (1).aoe2record
  map_id                   : 9
  rated                    : True
  num_players              : 2
  duration_min             : 45.45
  resign_player            : 1
  resign_time_min          : 45.45

=== Age-up times (player_num -> age -> minutes) ===
  Player 1: {'feudal': 6.6, 'castle': 17.58, 'imperial': 33.64}
  Player 2: {'feudal': 6.79, 'castle': 17.65, 'imperial': 29.9}

=== Leaderboard ratings (player_num -> Elo) ===
  player_num=0: 1893
  player_num=1: 1895

=== Header players ===
  Player 1: {'number': 1, 'type': 1, 'name': b'TheRealRuClEsHe', 'diplomacy': [0, 1, 4], 'civilization_id': 22, 'color_id': 0, 'position': {'x': 91.0, 'y': 43.0}}
  Player 2: {'number': 2, 'type': 1, 'name': b'z\xe9\x9d\x92\xe9\xb8\x9f127', 'diplomacy': [0, 4, 1], 'civilization_id': 53, 'color_id': 0, 'position': {'x': 19

## Section 3b — Build Per-Player Feature Rows

The batch parser will produce **one row per player per game** (2 rows per 1v1 game).

This cell assembles the flat feature dict that becomes a DataFrame row.

**Winner determination:**
- `resign_player` = player_num of the player who resigned = the loser
- The other player = winner
- `winner = 1` for the non-resigning player, `0` for the resigning player

**Leaderboard player_num note:**
The leaderboard stores entries keyed by `player_num`. In testing (Feb 2026 replays), `player_num=N` in the leaderboard aligns with game `player_number=N` from the header for player 1, but player 2 sometimes appears as `player_num=0` in the leaderboard. This needs verification across more replays before relying on direct key matching. Safe approach: match the two leaderboard entries to the two header players by elimination.

In [5]:
def build_player_rows(result):
    """Convert parse_replay() output into a list of flat feature dicts, one per player."""
    rows = []
    player_nums = sorted(result['header_players'].keys())
    
    # Match leaderboard ratings to players
    # Direct match by player_num where available; fill remaining by elimination
    lb = result['leaderboard_ratings']  # e.g. {1: 1787, 0: 1877}
    rating_by_player = {}
    for pnum in player_nums:
        if pnum in lb:
            rating_by_player[pnum] = lb[pnum]
    # If a player didn't match by direct key, assign the remaining rating
    assigned = set(rating_by_player.values())
    remaining_ratings = [v for k, v in lb.items() if v not in assigned]
    for pnum in player_nums:
        if pnum not in rating_by_player and remaining_ratings:
            rating_by_player[pnum] = remaining_ratings.pop(0)
    
    for pnum in player_nums:
        hp = result['header_players'][pnum]
        age_times = result['age_up_times'].get(pnum, {})
        
        won = 1 if result['resign_player'] is not None and pnum != result['resign_player'] else 0
        # If no resign recorded (e.g. disconnect, game still ongoing), win is unknown
        won = None if result['resign_player'] is None else won
        
        row = {
            'filepath':       result['filepath'],
            'player_num':     pnum,
            'player_name':    hp['name'].decode('utf-8', errors='replace'),
            'civilization_id': hp['civilization_id'],
            'map_id':         result['map_id'],
            'rated':          result['rated'],
            'duration_min':   result['duration_min'],
            'result':         won,           # 1 = won, 0 = lost, None = unknown
            'elo':            rating_by_player.get(pnum),
            'feudal_time_min':  age_times.get('feudal'),
            'castle_time_min':  age_times.get('castle'),
            'imperial_time_min': age_times.get('imperial'),
        }
        rows.append(row)
    return rows


rows = build_player_rows(result)
df_game = pd.DataFrame(rows)
print(df_game.to_string())

                                                                                                                           filepath  player_num      player_name  civilization_id  map_id  rated  duration_min  result   elo  feudal_time_min  castle_time_min  imperial_time_min
0  C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @2026.02.17 221325 (1).aoe2record           1  TheRealRuClEsHe               22       9   True         45.45       0  1895             6.60            17.58              33.64
1  C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @2026.02.17 221325 (1).aoe2record           2           z青鸟127               53       9   True         45.45       1  1893             6.79            17.65              29.90


## Section 4 — Inspect Field Structure

Check data types, null values, and consistency across two replays.

In [6]:
print('Field dtypes:')
print(df_game.dtypes)
print()
print('Null fields:')
print(df_game.isnull().sum())

Field dtypes:
filepath              object
player_num             int64
player_name           object
civilization_id        int64
map_id                 int64
rated                   bool
duration_min         float64
result                 int64
elo                    int64
feudal_time_min      float64
castle_time_min      float64
imperial_time_min    float64
dtype: object

Null fields:
filepath             0
player_num           0
player_name          0
civilization_id      0
map_id               0
rated                0
duration_min         0
result               0
elo                  0
feudal_time_min      0
castle_time_min      0
imperial_time_min    0
dtype: int64


In [7]:
# Parse a second 1v1 Arabia rated game to verify field structure is consistent
second_result = None
for filepath in reversed(replay_files):
    if filepath == REPLAY_PATH:
        continue
    try:
        r = parse_replay(filepath)
        if r['map_id'] == ARABIA_MAP_ID and r['rated'] and r['num_players'] == 2:
            second_result = r
            print(f'Second replay: {filepath.name}')
            break
    except Exception:
        continue

if second_result:
    rows2 = build_player_rows(second_result)
    df2 = pd.DataFrame(rows2)
    fields_1 = set(df_game.columns)
    fields_2 = set(df2.columns)
    print(f'Fields consistent across two games: {fields_1 == fields_2}')
    if fields_1 != fields_2:
        print(f'Only in game 1: {fields_1 - fields_2}')
        print(f'Only in game 2: {fields_2 - fields_1}')
    print()
    print('Second game rows:')
    print(df2.to_string())
else:
    print('Could not find a second parseable 1v1 Arabia rated game')

Second replay: MP Replay v101.103.31214.0 @2026.02.17 214339 (1).aoe2record
Fields consistent across two games: True

Second game rows:
                                                                                                                           filepath  player_num      player_name  civilization_id  map_id  rated  duration_min  result   elo  feudal_time_min  castle_time_min imperial_time_min
0  C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @2026.02.17 214339 (1).aoe2record           1  TheRealRuClEsHe                6       9   True         29.99       1  2094             6.66            17.19              None
1  C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @2026.02.17 214339 (1).aoe2record           2                𝝿               33       9   True         29.99       0  1868             6.54            17.44              None


## Section 5 — Required Feature Checklist

All 5 Required features must be FOUND to proceed without further fallback.
Run Section 3b first so `df_game` exists.

In [8]:
required_checks = [
    ('Win/loss outcome       [REQ]', 'result',           "1=won, 0=lost from RESIGN action"),
    ('Player Elo             [REQ]', 'elo',               "From POSTGAME leaderboard id=3"),
    ('Map name/id            [REQ]', 'map_id',            "de['rms_map_id'] — Arabia=9"),
    ('Feudal age timing      [REQ]', 'feudal_time_min',   "RESEARCH action tech_id=101"),
    ('Rated game             [REQ]', 'rated',             "de['rated'] boolean"),
    ('Game duration               ', 'duration_min',      "POSTGAME world_time or SYNC sum"),
    ('Castle age timing           ', 'castle_time_min',   "RESEARCH action tech_id=102"),
    ('Imperial age timing         ', 'imperial_time_min', "RESEARCH action tech_id=103"),
    ('Civilization                ', 'civilization_id',   "header players[n]['civilization_id']"),
]

print(f'{"Feature":<40} {"Status":<8} Source')
print('-' * 80)
all_required_present = True
for label, field, source in required_checks:
    required = '[REQ]' in label
    present = field in df_game.columns and df_game[field].notna().any()
    status = 'FOUND' if present else 'MISSING'
    if required and not present:
        all_required_present = False
        status = '!! MISSING !!'
    print(f'{label:<40} {status:<14} {source}')

print()
if all_required_present:
    print('All required features FOUND — mgz-fast is sufficient. Proceed to 02_bulk_parsing.ipynb.')
else:
    print('WARNING: One or more required features are missing. Investigate before proceeding.')

Feature                                  Status   Source
--------------------------------------------------------------------------------
Win/loss outcome       [REQ]             FOUND          1=won, 0=lost from RESIGN action
Player Elo             [REQ]             FOUND          From POSTGAME leaderboard id=3
Map name/id            [REQ]             FOUND          de['rms_map_id'] — Arabia=9
Feudal age timing      [REQ]             FOUND          RESEARCH action tech_id=101
Rated game             [REQ]             FOUND          de['rated'] boolean
Game duration                            FOUND          POSTGAME world_time or SYNC sum
Castle age timing                        FOUND          RESEARCH action tech_id=102
Imperial age timing                      FOUND          RESEARCH action tech_id=103
Civilization                             FOUND          header players[n]['civilization_id']

All required features FOUND — mgz-fast is sufficient. Proceed to 02_bulk_parsing.ipynb.


## Section 6 — Feature Documentation Table

**Fill this in after running Section 3b.** One row per extracted field. This is the authoritative feature reference for Weeks 2–3.

| field_name | data_type | aoe2_meaning | expected_range | keep_for_model | notes |
|---|---|---|---|---|---|
| result | int (0/1/None) | Did this player win? Used as training label y | 0 or 1 | Required (label) | None if no resign recorded |
| elo | int | Player's ranked 1v1 RM Elo at time of game | 800–2200 | Required (grouping) | From POSTGAME leaderboard id=3 |
| map_id | int | Which map was played (9=Arabia) | 9 for Arabia | Filter only | |
| rated | bool | Was this a ranked game? | True/False | Filter only | |
| duration_min | float | How long the game lasted | 5–90 min | Yes | Games < 8 min = filtered in Week 2 |
| feudal_time_min | float | When player clicked up to Feudal Age | 8–30 min | Yes | Lower = faster = generally better |
| castle_time_min | float | When player clicked up to Castle Age | 14–40 min | Yes | Lower = faster |
| imperial_time_min | float | When player clicked up to Imperial Age | 22–60 min | Maybe | Many games end before Imp |
| civilization_id | int | Which civ the player used | 1–45+ | Maybe | One-hot encode in Week 2 |
| civilization_id | int | Civ picked | int | Maybe | |
| player_name | str | In-game username | string | No | Not a feature — for debugging only |

**Additional notes on what is NOT available:**
- Resources collected over time — not in .aoe2record
- Villager gather rates — not in .aoe2record
- Idle villager time — not directly available from mgz-fast (would require scanning all unit actions, not feasible for Week 1)
- Military unit idle time — same limitation

In [9]:
# Auto-generate documentation scaffold from the actual output
doc = pd.DataFrame({
    'field_name':    df_game.columns,
    'sample_p1':     df_game.iloc[0].values,
    'sample_p2':     df_game.iloc[1].values,
    'dtype':         df_game.dtypes.values,
    'aoe2_meaning':  '',
    'expected_range': '',
    'keep_for_model': '',
    'notes': ''
})
doc

,field_name,sample_p1,sample_p2,dtype,aoe2_meaning,expected_range,keep_for_model,notes
0,filepath,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @...,C:\Users\liher\Games\Age of Empires 2 DE\76561198151543542\savegame\MP Replay v101.103.31214.0 @...,object,,,,
1,player_num,1,2,int64,,,,
2,player_name,TheRealRuClEsHe,z青鸟127,object,,,,
3,civilization_id,22,53,int64,,,,
4,map_id,9,9,int64,,,,
5,rated,True,True,bool,,,,
6,duration_min,45.45,45.45,float64,,,,
7,result,0,1,int64,,,,
8,elo,1895,1893,int64,,,,
9,feudal_time_min,6.6,6.79,float64,,,,


## Section 7 — mgz-fast Sufficiency Assessment

After completing Sections 3–5, fill in this assessment before moving to `02_bulk_parsing.ipynb`.

**Required fields checklist:**
- [ ] Feudal/Castle/Imperial age-up timing: `feudal_time_min / castle_time_min / imperial_time_min` via RESEARCH tech IDs
- [ ] Win/loss outcome: `result` field via RESIGN action
- [ ] Player Elo: `elo` field via POSTGAME leaderboard id=3
- [ ] Map type: `map_id` field from `de['rms_map_id']` (Arabia=9)
- [ ] Rated game: `rated` field from `de['rated']`

**Known limitations vs. original scope:**
- Villager idle time: NOT extractable from mgz-fast without scanning every unit action (too slow/complex for MVP)
- Idle military time: Same limitation — defer to v0.2
- Opening strategy label: AgeAlyser_2 detected this as a categorical; not available in mgz-fast — drop from MVP feature set

**Decision:**
mgz-fast IS sufficient for the 5 required features. Villager idle time is deferred to v0.2. Proceed with mgz-fast as the sole parser.

---

**Assessment:** *(write here after running Section 5)*

**Conclusion:** mgz-fast is sufficient. All 5 required fields confirmed present.
Next step: `02_bulk_parsing.ipynb`